In [ ]:
import logging

import shapely

import openeo
import openeo.processes

logging.basicConfig(level=logging.INFO)

In [ ]:
connection = openeo.connect("openeo.dataspace.copernicus.eu")

In [ ]:
connection.authenticate_oidc()

# Parameters

In [ ]:
spatial_extent = {
    "west": 30.5503711040000994,
    "south": 1.0709279050000799,
    "east": 31.2229521229999989,
    "north": 1.5469373050000299,
}
temporal_extent = "2020-01-01"

In [ ]:
# very small test AOI
# x = 30.7
# y = 1.3
# delta = 0.1
# spatial_extent = {
#     "west": x,
#     "south": y - delta,
#     "east": x + delta,
#     "north": y,
# }

In [ ]:
# spatial extent as dict of Polygon geometry
spatial_extent = shapely.geometry.mapping(
    shapely.box(
        xmin=spatial_extent["west"],
        ymin=spatial_extent["south"],
        xmax=spatial_extent["east"],
        ymax=spatial_extent["north"],
    )
)

In [ ]:
# minimum canopy cover to be considered forest
canopy_cover_threshold = 30

# minimum likelihood to be considered natural forest
natural_forest_threshold = 0.08

# minimum connected area to be considered forest (m^2)
min_connected_area = 10000

# Script

In [ ]:
# collect outputs as we go, to built a multi-result process graph
process_graph_results = []

In [ ]:
# TODO: update this to permanent location
natural_forest_stac = "https://s3.waw3-2.cloudferro.com/swift/v1/leon-p6/natural-forest-float32/item.json"

In [ ]:
# Natural Forests of the World 2020
# EPSG:32636 = UTM zone 36N
# dims: ['x', 'y', 'bands']
natural_forest = connection.load_stac(
    url=natural_forest_stac,
    spatial_extent=spatial_extent,
    temporal_extent=temporal_extent,
    bands=["B0"],
)

In [ ]:
# connection.describe_collection("ESA_WORLDCOVER_10M_2020_V1")

In [ ]:
# https://openeofed.dataspace.copernicus.eu/?discover=0&collection=ESA_WORLDCOVER_10M_2020_V1
# 10 m resolution
# EPSG:4326
# openEO backend: cdse
# ⚠️ on terrascope (and federated) backend this collection has different bands!
esa_worldcover = connection.load_collection(
    "ESA_WORLDCOVER_10M_2020_V1",
    spatial_extent=spatial_extent,
    temporal_extent=temporal_extent,
    bands=["MAP"],
)

In [ ]:
# remove the time dimension
esa_worldcover = esa_worldcover.reduce_dimension("t", reducer=openeo.processes.first)

In [ ]:
# connection.describe_collection("CLMS_TCD_PANTROPICAL_10M_YEARLY_V1")

In [ ]:
# tree cover density
# https://openeofed.dataspace.copernicus.eu/?discover=0&collection=CLMS_TCD_PANTROPICAL_10M_YEARLY_V1
# 10 m resolution
# EPSG:4326
# openEO backend: cdse
tree_cover_density = connection.load_collection(
    "CLMS_TCD_PANTROPICAL_10M_YEARLY_V1",
    spatial_extent=spatial_extent,
    temporal_extent=temporal_extent,
    bands=["map"],
)

In [ ]:
# remove the time dimension
tree_cover_density = tree_cover_density.reduce_dimension(
    "t", reducer=openeo.processes.first
)

In [ ]:
# align all 3 datasets in UTM zone 36N
esa_worldcover = esa_worldcover.resample_cube_spatial(natural_forest, method="near")
tree_cover_density = tree_cover_density.resample_cube_spatial(
    natural_forest, method="near"
)

In [ ]:
# make sure band dimension has consistent labels
esa_worldcover = esa_worldcover.rename_labels(dimension="bands", target=["B0"])
tree_cover_density = tree_cover_density.rename_labels(dimension="bands", target=["B0"])

In [ ]:
process_graph_results.append(
    natural_forest.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0000_natural_forest",
        },
    )
)
process_graph_results.append(
    esa_worldcover.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0000_esa_worldcover",
        },
    )
)
process_graph_results.append(
    tree_cover_density.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0000_tree_cover_density",
        },
    )
)

In [ ]:
# mask, 1 = natural forest
forest_baseline = (
    (
        esa_worldcover == 10  # class 10 = tree cover
    )
    & (tree_cover_density >= canopy_cover_threshold)
    & (
        tree_cover_density <= 100  # values above 100 = unclassifiable / no_data
    )
    & (natural_forest > natural_forest_threshold)
)

In [ ]:
process_graph_results.append(
    forest_baseline.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0010_forest_baseline",
        },
    )
)

In [ ]:
connectivity_udf = openeo.UDF.from_file(
    "../udf/connectivity_mask.py",
    runtime="Python",
    version="3.11",
    context={
        "pixel_area": 10 * 10,
        "min_connected_area": min_connected_area,
    },
)

In [ ]:
# mask where 1 = small region to be excluded
small_region_mask = forest_baseline.apply_neighborhood(
    connectivity_udf,
    size=[
        {"dimension": "x", "value": 256, "unit": "px"},
        {"dimension": "y", "value": 256, "unit": "px"},
    ],
    # overlap needs to be big enough the reasonably allow for min_pixels
    overlap=[
        {"dimension": "x", "value": 32, "unit": "px"},
        {"dimension": "y", "value": 32, "unit": "px"},
    ],
)

In [ ]:
process_graph_results.append(
    small_region_mask.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0020_small_region_mask",
        },
    )
)

In [ ]:
forest_baseline = forest_baseline & ~small_region_mask

In [ ]:
process_graph_results.append(
    forest_baseline.save_result(
        format="GTiff",
        options={
            "filename_prefix": "0030_forest_baseline",
        },
    )
)

# Run job

In [ ]:
multi_result = openeo.MultiResult(process_graph_results)
job = multi_result.create_job()

In [ ]:
job.start_and_wait()

In [ ]:
import datetime
import time

while True:
    try:
        status = job.status()
        print(datetime.datetime.now().isoformat(), status)
    except Exception as e:
        print(datetime.datetime.now().isoformat(), e)
        time.sleep(2)
    else:
        if status != "running":
            break
        time.sleep(10)

In [ ]:
import json

with open("logs.json", "w") as f:
    json.dump(job.logs(), f, indent=2)

In [ ]:
results = job.get_results()

In [ ]:
!mkdir -p output/
!rm -r output/

In [ ]:
results.download_files("output/")